Reweighing (Kamiran & Calders, 2012): fairness evaluation on Adult, German Credit and COMPAS

This notebook applies the **Reweighing** pre-processing method to three datasets and compares the fairness measures before and after mitigation.


In [1]:
# Install AIF360 (the library with the Reweighing implementation).
%pip install -q aif360

In [2]:
# Reweighing (Kamiran & Calders, 2012) on Adult, German Credit and COMPAS
# The reweighing step comes from IBM's AIF360 library. Everything else is plain pandas / scikit-learn.

import warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from aif360.sklearn.preprocessing import Reweighing

warnings.filterwarnings("ignore")

N_SEEDS = 10          # number of random train/test splits
TEST_SIZE = 0.3       # 70% train, 30% test
K_NEIGHBOURS = 5      # k for the consistency measure (as in Zemel et al., 2013)

## Settings and data
Each loader returns three things: the features **X** (without the protected attribute), the label **y** (1 = favourable outcome) and the protected attribute **s** (1 = privileged group).

| Dataset | Protected attribute (privileged) | Favourable label |
|---|---|---|
| Adult | sex (male) | income > 50K |
| German Credit | age (25 or older) | good credit |
| COMPAS | race (Caucasian) | no re-offence within two years |

In [3]:
BASE = "https://raw.githubusercontent.com/"


def load_adult():
    cols = ["age", "workclass", "fnlwgt", "education", "education_num", "marital_status",
            "occupation", "relationship", "race", "sex", "capital_gain", "capital_loss",
            "hours_per_week", "native_country", "income"]
    df = pd.read_csv(BASE + "jbrownlee/Datasets/master/adult-all.csv", names=cols,
                     na_values="?", skipinitialspace=True).dropna()
    y = (df["income"] == ">50K").astype(int)          # favourable label: high income
    s = (df["sex"] == "Male").astype(int)             # privileged group: male
    X = df.drop(columns=["income", "sex", "fnlwgt"])  # fnlwgt removed, as in the paper
    return X, y, s


def load_german():
    cols = ["status", "duration", "credit_history", "purpose", "amount", "savings",
            "employment", "installment_rate", "personal_status", "other_debtors",
            "residence_since", "property", "age", "other_plans", "housing",
            "existing_credits", "job", "people_liable", "telephone", "foreign_worker", "credit"]
    df = pd.read_csv(BASE + "jbrownlee/Datasets/master/german.csv", names=cols)
    y = (df["credit"] == 1).astype(int)               # favourable label: good credit
    s = (df["age"] >= 25).astype(int)                 # privileged group: age 25 or older
    X = df.drop(columns=["credit", "age"])
    return X, y, s


def load_compas():
    df = pd.read_csv(BASE + "propublica/compas-analysis/master/compas-scores-two-years.csv")
    # standard ProPublica filtering
    df = df[(df.days_b_screening_arrest <= 30) & (df.days_b_screening_arrest >= -30)
            & (df.is_recid != -1) & (df.c_charge_degree != "O") & (df.score_text != "N/A")]
    y = (df["two_year_recid"] == 0).astype(int)       # favourable label: no re-offence
    s = (df["race"] == "Caucasian").astype(int)       # privileged group: Caucasian
    X = df[["sex", "age", "juv_fel_count", "juv_misd_count", "juv_other_count",
            "priors_count", "c_charge_degree"]]
    return X, y, s


DATASETS = {"Adult (sex)": load_adult, "German (age)": load_german, "COMPAS (race)": load_compas}

## Fairness measures
- **DP_Diff** = P(Ŷ=1 | unprivileged) − P(Ŷ=1 | privileged). 0 is fair. A negative value means the unprivileged group gets the good outcome less often.
- **DIR** = P(Ŷ=1 | unprivileged) / P(Ŷ=1 | privileged). 1 is fair. Below 0.8 fails the "four-fifths rule".
- **Consistency** (Zemel et al., 2013): do similar people get similar outcomes? 1 means each person gets the same outcome as their 5 nearest neighbours.

In [4]:
def dp_diff(pred, s):
    """P(favourable | unprivileged) - P(favourable | privileged). 0 is fair, negative = unprivileged worse off."""
    return pred[s == 0].mean() - pred[s == 1].mean()


def dir_ratio(pred, s):
    """P(favourable | unprivileged) / P(favourable | privileged). 1 is fair, the 80% rule asks for >= 0.8."""
    return pred[s == 0].mean() / pred[s == 1].mean()


def consistency(X_scaled, labels, k=K_NEIGHBOURS):
    """1 - mean |label_i - mean label of its k nearest neighbours| (Zemel et al., 2013). 1 is perfectly consistent."""
    nn = NearestNeighbors(n_neighbors=k + 1).fit(X_scaled)
    idx = nn.kneighbors(X_scaled, return_distance=False)[:, 1:]   # drop the point itself
    return 1 - np.mean(np.abs(labels - labels[idx].mean(axis=1)))


def encode(X_train, X_test):
    """One-hot encode categorical columns and standardise everything, fitted on the training part only."""
    both = pd.get_dummies(pd.concat([X_train, X_test]), drop_first=False).astype(float)
    tr, te = both.iloc[:len(X_train)], both.iloc[len(X_train):]
    sc = StandardScaler().fit(tr)
    return sc.transform(tr), (sc.transform(te) if len(te) else None)

## The experiment
For each dataset: first the measures on the original labels, then 10 random stratified splits. In each split the weights are computed on the training part only, and logistic regression is trained once without and once with the weights. Following the paper (assumption A2), the protected attribute is **not** used as a feature of the classifier.

For the first split we also check that AIF360's weights equal the formula in the paper, W(s, c) = P(S=s) · P(Class=c) / P(S=s, Class=c), and that the weighted training data has a DP_Diff of 0.

In [5]:
rows, before_rows, weight_rows = [], [], []

for name, loader in DATASETS.items():
    X, y, s = loader()
    X, y, s = X.reset_index(drop=True), y.to_numpy(), s.to_numpy()

    # Step 2: measures on the original labels, before any mitigation or classifier
    X_all, _ = encode(X, X.iloc[:0])
    before_rows.append({"Dataset": name, "n": len(y), "Fav. rate": y.mean(),
                        "Priv. share": s.mean(),
                        "DP_Diff": dp_diff(y, s), "DIR": dir_ratio(y, s),
                        "Consistency": consistency(X_all, y)})

    for seed in range(N_SEEDS):
        # Step 3: stratified split, stratified on label AND group so both keep their shares
        strata = 2 * s + y
        X_tr, X_te, y_tr, y_te, s_tr, s_te = train_test_split(
            X, y, s, test_size=TEST_SIZE, stratify=strata, random_state=seed)
        Xtr, Xte = encode(X_tr, X_te)

        # Step 4: reweighing on the TRAINING data only (pre-processing)
        # AIF360's sklearn API reads the protected attribute from the index
        Xtr_df = pd.DataFrame(Xtr, index=pd.Index(s_tr, name="prot"))
        _, w = Reweighing(prot_attr="prot").fit_transform(Xtr_df, pd.Series(y_tr, index=Xtr_df.index))
        w = np.asarray(w)

        if seed == 0:   # keep the four weights to check them against the paper's formula
            for g in (0, 1):
                for c in (0, 1):
                    m = (s_tr == g) & (y_tr == c)
                    paper_w = (s_tr == g).mean() * (y_tr == c).mean() / m.mean()
                    weight_rows.append({"Dataset": name, "Group": "priv" if g else "unpriv",
                                        "Label": c, "AIF360 weight": w[m][0], "Paper formula": paper_w})
            wdp = (np.average(y_tr[s_tr == 0], weights=w[s_tr == 0])
                   - np.average(y_tr[s_tr == 1], weights=w[s_tr == 1]))
            weight_rows.append({"Dataset": name, "Group": "weighted train DP_Diff",
                                "Label": "", "AIF360 weight": wdp, "Paper formula": 0.0})

        # Step 5: classifier without and with reweighing; the protected attribute is NOT a feature (paper, A2)
        for method, sw in (("No mitigation", None), ("Reweighing", w)):
            clf = LogisticRegression(max_iter=2000).fit(Xtr, y_tr, sample_weight=sw)
            pred = clf.predict(Xte)
            rows.append({"Dataset": name, "Method": method, "Seed": seed,
                         "DP_Diff": dp_diff(pred, s_te), "DIR": dir_ratio(pred, s_te),
                         "Consistency": consistency(Xte, pred),
                         "Accuracy": accuracy_score(y_te, pred),
                         "Balanced acc.": balanced_accuracy_score(y_te, pred)})

before = pd.DataFrame(before_rows)
res = pd.DataFrame(rows)
weights = pd.DataFrame(weight_rows)

pd.set_option("display.width", 200)
print("Before mitigation (original labels, full dataset)\n", before.round(3).to_string(index=False))
print("\nReweighing weights, seed 0 (AIF360 vs paper formula)\n", weights.round(4).to_string(index=False))
summary = res.groupby(["Dataset", "Method"]).agg(["mean", "std"]).drop(columns="Seed")
print("\nTest set, mean and sd over", N_SEEDS, "splits\n", summary.round(3).to_string())

import os
os.makedirs("results", exist_ok=True)
before.to_csv("results/before.csv", index=False)
res.to_csv("results/results_per_seed.csv", index=False)
summary.to_csv("results/results_summary.csv")

Before mitigation (original labels, full dataset)
       Dataset     n  Fav. rate  Priv. share  DP_Diff   DIR  Consistency
  Adult (sex) 45222      0.248        0.675   -0.199 0.363        0.792
 German (age)  1000      0.700        0.851   -0.129 0.821        0.666
COMPAS (race)  6172      0.545        0.341   -0.097 0.840        0.625

Reweighing weights, seed 0 (AIF360 vs paper formula)
       Dataset                  Group Label  AIF360 weight  Paper formula
  Adult (sex)                 unpriv     0         0.8485         0.8485
  Adult (sex)                 unpriv     1         2.1825         2.1825
  Adult (sex)                   priv     0         1.0940         1.0940
  Adult (sex)                   priv     1         0.7931         0.7931
  Adult (sex) weighted train DP_Diff              -0.0000         0.0000
 German (age)                 unpriv     0         0.7326         0.7326
 German (age)                 unpriv     1         1.1855         1.1855
 German (age)         

## Extra check: what if the classifier may use the protected attribute?
On COMPAS, Reweighing barely changes the result. Here we test the explanation: the model cannot see race, and race is hard to predict from the other features, so the weights cannot shift one group's predictions. If we allow the protected attribute as a feature (this breaks assumption A2 of the paper, so it is only a check), Reweighing should close the gap.

In [6]:
extra = []
for name, loader in DATASETS.items():
    X, y, s = loader()
    X, y, s = X.reset_index(drop=True), y.to_numpy(), s.to_numpy()
    for seed in range(N_SEEDS):
        X_tr, X_te, y_tr, y_te, s_tr, s_te = train_test_split(
            X, y, s, test_size=TEST_SIZE, stratify=2 * s + y, random_state=seed)
        # this time the protected attribute is added as an extra feature
        Xtr, Xte = encode(X_tr.assign(prot=s_tr), X_te.assign(prot=s_te))
        # the same weights as before, computed directly with the paper's formula
        w = np.array([(s_tr == g).mean() * (y_tr == c).mean() / ((s_tr == g) & (y_tr == c)).mean()
                      for g, c in zip(s_tr, y_tr)])
        for method, sw in (("No mitigation", None), ("Reweighing", w)):
            pred = LogisticRegression(max_iter=2000).fit(Xtr, y_tr, sample_weight=sw).predict(Xte)
            extra.append({"Dataset": name, "Method": method, "DP_Diff": dp_diff(pred, s_te),
                          "DIR": dir_ratio(pred, s_te), "Accuracy": accuracy_score(y_te, pred)})
print("Protected attribute used as a feature (mean over 10 splits)")
print(pd.DataFrame(extra).groupby(["Dataset", "Method"]).mean().round(3))

# How well can the protected attribute be predicted from the other features?
# A high value means the features carry a lot of information about the group (strong proxies).
from sklearn.model_selection import cross_val_score
for name, loader in DATASETS.items():
    X, y, s = loader()
    Xa, _ = encode(X.reset_index(drop=True), X.iloc[:0])
    score = cross_val_score(LogisticRegression(max_iter=2000, class_weight="balanced"), Xa, s,
                            cv=5, scoring="balanced_accuracy").mean()
    print(f"{name}: protected attribute predictable from the features, balanced accuracy = {score:.3f}")

Protected attribute used as a feature (mean over 10 splits)
                             DP_Diff    DIR  Accuracy
Dataset       Method                                 
Adult (sex)   No mitigation   -0.183  0.303     0.845
              Reweighing      -0.087  0.601     0.840
COMPAS (race) No mitigation   -0.185  0.758     0.681
              Reweighing       0.008  1.014     0.664
German (age)  No mitigation   -0.160  0.796     0.752
              Reweighing       0.001  1.001     0.748
Adult (sex): protected attribute predictable from the features, balanced accuracy = 0.861
German (age): protected attribute predictable from the features, balanced accuracy = 0.707
COMPAS (race): protected attribute predictable from the features, balanced accuracy = 0.608


In [7]:
# Software versions used for this run
import sys, sklearn, aif360
print("Python", sys.version.split()[0], "| numpy", np.__version__, "| pandas", pd.__version__,
      "| scikit-learn", sklearn.__version__, "| aif360", aif360.__version__)

Python 3.13.18 | numpy 2.4.6 | pandas 2.2.3 | scikit-learn 1.6.1 | aif360 0.6.1
